In [ ]:
import pandas as pd
from collections import deque

In [ ]:
twins = {}

with open('/mnt/project/DNM/twins_pairs.txt') as f:
    
    for row in f:
        
        row = row.strip()
        row = row.split(' ')
        twin1 = int(row[0])
        twin2 = int(row[1])
        twins[twin1] = twin1
        twins[twin2] = twin1

# Trio families

In [ ]:
trios_pairs = {}

trios_ukb_df = pd.read_csv('/mnt/project/DNM/trios_pairs.txt', sep = ' ', header = None)

for i, row in trios_ukb_df.iterrows():
    
    o = int(row[0])
    f = int(row[1])
    m = int(row[2])
    
    if o in twins and twins[o] != o:
        continue
        
    trios_pairs[(o, f, m)] = True

In [ ]:
print(len(trios_ukb_df), len(trios_pairs))

In [ ]:
pars_off = {}

for o, f, m in trios_pairs:
    if (f, m) not in pars_off:
        pars_off[(f, m)] = []
    pars_off[(f, m)].append(o)
    
off_pars = {}

for o, f, m in trios_pairs:
    off_pars[o] = (f, m)

In [ ]:
cnts = {}

for f, m in pars_off:
    if len(pars_off[(f, m)]) not in cnts:
        cnts[len(pars_off[(f, m)])] = 0
    cnts[len(pars_off[(f, m)])] += 1
    
cnts = dict(sorted(cnts.items(), key = lambda x: (x[0])))
        
print(len(pars_off), cnts)

In [ ]:
trios_inds = {}

trios_ukb_dnms_df = pd.read_csv('/mnt/project/DNM/trios/out/trios_diffs_info.txt', sep = '\t')

for _, row in trios_ukb_dnms_df.iterrows():
    
    ind_idx = row['IID']
    ind = int(ind_idx.split('_')[0])
    trios_inds[ind] = True

In [ ]:
fams = {}
id_fam = {}
ids = 0

for f, m in pars_off:
    for o in pars_off[(f, m)]:
        if o in trios_inds:
            if ids not in fams:
                fams[ids] = []
                id_fam[ids] = (f, m)
            fams[ids].append(o)
    if ids in fams:
        ids += 1
        
print(len(fams))

# Sibs families

In [ ]:
sibs_pairs = {}

sibs_ukb_df = pd.read_csv('/mnt/project/DNM/sibs_pairs.txt', sep = ' ', header = None)

for i, row in sibs_ukb_df.iterrows():
    
    sib1 = int(row[0])
    sib2 = int(row[1])
    
    if sib1 in twins and twins[sib1] != sib1:
        continue
    if sib2 in twins and twins[sib2] != sib2:
        continue
        
    sibs_pairs[(sib1, sib2)] = True

In [ ]:
print(len(sibs_ukb_df), len(sibs_pairs))

In [ ]:
graph = {}

for sib1, sib2 in sibs_pairs:
    if sib1 not in graph:
        graph[sib1] = []
    graph[sib1].append(sib2)
    
    if sib2 not in graph:
        graph[sib2] = []
    graph[sib2].append(sib1)

In [ ]:
fams_temp = {}
ids_temp = 0

visited = set()

for sib in graph:
    
    if sib in visited:
        continue
    
    queue = deque([sib])
    visited.add(sib)
    fam = set([sib])
    
    while queue:
        
        sib1 = queue.popleft()
        
        for sib2 in graph[sib1]:
            if sib2 not in visited:
                queue.append(sib2)
                visited.add(sib2)
                fam.add(sib2)
    
    fams_temp[ids_temp] = fam
    ids_temp += 1

In [ ]:
cnts = {}

for i in fams_temp:
    if len(fams_temp[i]) not in cnts:
        cnts[len(fams_temp[i])] = 0
    cnts[len(fams_temp[i])] += 1
    
cnts = dict(sorted(cnts.items(), key = lambda x: (x[0])))
    
print(len(fams_temp), cnts)

In [ ]:
sibs = {}

sibs_ukb_dnms_df = pd.read_csv('/mnt/project/DNM/sibs/out/sibs_diffs_info.txt', sep = '\t')

for _, row in sibs_ukb_dnms_df.iterrows():
    
    ind_idx = row['IID']
    sibs[ind_idx] = True

In [ ]:
sibs_inds_qc = {}
sibs_pairs_qc = {}

sibs_ukb_df = pd.read_csv('/mnt/project/DNM/sibs_pairs.txt', sep = ' ', header = None)

for i, row in sibs_ukb_df.iterrows():
    
    sib1 = int(row[0])
    sib2 = int(row[1])
    
    if sib1 in twins and twins[sib1] != sib1:
        continue
    if sib2 in twins and twins[sib2] != sib2:
        continue
    
    b = int(i/100)
    p = i%100
            
    sib1_idx = f'{sib1}_b{b}_p{p}'
    sib2_idx = f'{sib2}_b{b}_p{p}'
    
    if sib1_idx in sibs or sib2_idx in sibs:
        sibs_inds_qc[sib1] = True
        sibs_inds_qc[sib2] = True
        sibs_pairs_qc[(sib1, sib2)] = True

In [ ]:
for i in fams_temp:
    for sib in fams_temp[i]:
        if sib in sibs_inds_qc and sib not in off_pars:
            if ids not in fams:
                fams[ids] = []
                id_fam[ids] = 'NA'
            fams[ids].append(sib)
    if ids in fams:
        ids += 1
        
print(len(fams))

# Families

In [ ]:
df_fams = pd.DataFrame({'ID': fams.keys(), 'INDS': [','.join(map(str, inds)) for inds in fams.values()]})
df_fams.to_csv('ukb_families.csv', index = False)